# decoNET: train the decoration specialist

This notebook trains **deconet from randomly initialized weights**. It does not download a base model, call an inference API, or require a provider key.

**what to expect:** the starter corpus is synthetic and template-generated. This is the first experiment to prove the model can learn the JSON task and improve beyond its random-init baseline. It will not immediately be a strong language model or prove real Geometry Dash expertise.

## before running

1. Create a GitHub repo named `gd-ai-lab` and push this project to it (see README).
2. Set a GPU runtime in Colab or Kaggle.
3. In the next cell, replace `YOUR_USERNAME` with the account that owns the repo. The repo must be public for unauthenticated `git clone`.
4. Run cells in order. Save the downloaded ZIP somewhere safe. Do not commit the `.pt` checkpoint to GitHub.


In [3]:
import os, sys, subprocess, shutil, json, zipfile, time
from pathlib import Path
import torch

REPO_URL = "https://github.com/Chayce22315/gd-ai-lab.git"  # edit this once
WORKSPACE = Path('/kaggle/working' if Path('/kaggle/working').exists() else '/content')
PROJECT = WORKSPACE / 'gd-ai-lab'
RUNS = WORKSPACE / 'gd-ai-lab-runs'

if 'YOUR_USERNAME' in REPO_URL:
    raise ValueError('Edit REPO_URL and replace YOUR_USERNAME with your GitHub username.')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT, check=True)
os.chdir(PROJECT)
print('project:', PROJECT)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('vram_gib:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError('No GPU detected. Enable a GPU runtime in notebook settings before training.')


ValueError: Edit REPO_URL and replace YOUR_USERNAME with your GitHub username.

## install the project

The repo uses PyTorch only for the model and training loop. A fixed byte tokenizer avoids extra tokenizer downloads.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT)], check=True)
print('editable install complete')


## build training data

The starter generator makes reproducible synthetic prompts and JSON completions. Inspect `data/generated/manifest.json` and a few rows before training if you want to see exactly what it is learning.

In [ ]:
subprocess.run([
    sys.executable, 'scripts/build_dataset.py',
    '--output-dir', 'data/generated',
    '--samples-per-model', '1200',
    '--seed', '42',
], cwd=PROJECT, check=True)
print((PROJECT / 'data/generated/manifest.json').read_text())


## train deconet

The starting config is intentionally small: 192 hidden dimensions, 4 Transformer blocks, 6 attention heads. For an 8 GB GPU, begin with batch size 2 and gradient accumulation 4. If you get CUDA out-of-memory, use batch size 1 before changing anything else.

In [ ]:
MODEL = 'deconet'
OUTPUT = RUNS / MODEL
OUTPUT.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, '-m', 'gdai.train',
    '--model', MODEL,
    '--train-file', str(PROJECT / f'data/generated/{MODEL}_train.jsonl'),
    '--val-file', str(PROJECT / f'data/generated/{MODEL}_val.jsonl'),
    '--output-dir', str(OUTPUT),
    '--epochs', '5',
    '--batch-size', '2',
    '--gradient-accumulation', '4',
    '--max-seq-len', '1536',
    '--d-model', '192',
    '--layers', '4',
    '--heads', '6',
    '--learning-rate', '0.0003',
], cwd=PROJECT, check=True)
print('best checkpoint:', OUTPUT / 'checkpoint_best.pt')


## evaluate and generate one example

The metrics distinguish JSON-valid rate from schema-valid rate. Neither one demonstrates that a level is playable or visually rate-worthy.

In [ ]:
REPORT = OUTPUT / 'evaluation.json'
subprocess.run([
    sys.executable, '-m', 'gdai.evaluate',
    '--model', 'deconet',
    '--checkpoint', str(OUTPUT / 'checkpoint_best.pt'),
    '--data', str(PROJECT / 'data/generated/deconet_val.jsonl'),
    '--limit', '5',
    '--max-new-tokens', '1200',
    '--report', str(REPORT),
], cwd=PROJECT, check=True)


In [ ]:
TASK = 'decoration_plan'
REQUEST = 'modern glow on an abandoned orbital greenhouse; cyan and amber palette; calm opening, intricate build, explosive drop; prioritize depth and readable gameplay'
subprocess.run([
    sys.executable, '-m', 'gdai.generate',
    '--model', 'deconet',
    '--checkpoint', str(OUTPUT / 'checkpoint_best.pt'),
    '--task', TASK,
    '--prompt', REQUEST,
    '--max-new-tokens', '1200',
    '--json-output', str(OUTPUT / 'sample_plan.json'),
    '--svg-output', str(OUTPUT / 'decoration_preview.svg'),
], cwd=PROJECT, check=True)
from IPython.display import SVG, display
display(SVG(filename=str(OUTPUT / 'decoration_preview.svg')))


## export your run

Download the ZIP and keep it outside Git. It includes the best/last checkpoints, the training history, config, and eval report.

In [ ]:
BUNDLE = WORKSPACE / f'{MODEL}-training-artifacts.zip'
with zipfile.ZipFile(BUNDLE, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT.rglob('*'):
        if p.is_file():
            zf.write(p, p.relative_to(OUTPUT.parent))
    manifest = PROJECT / 'data/generated/manifest.json'
    if manifest.exists(): zf.write(manifest, 'data_manifest.json')
print('artifact bundle:', BUNDLE, 'bytes:', BUNDLE.stat().st_size)
try:
    from google.colab import files
    files.download(str(BUNDLE))
except ImportError:
    print('Kaggle: use the notebook Output panel to download', BUNDLE)


## what to inspect next

- Did validation loss improve compared with the random-init baseline in `training_history.jsonl`?
- What share of held-out generations are valid JSON?
- Did schema-valid rate improve?
- Does the result generalize to prompts that were not produced by the same synthetic templates? (That requires a separately written real-world evaluation set.)

If the model still emits invalid JSON, do not assume it's smart because one generation looks promising. Increase high-quality curated examples, train longer, and re-evaluate. Store checkpoints outside GitHub. For training from scratch, data quality and number of useful tokens are major limits.